# Enterprise RAG System with RBAC & Guardrails

A **Retrieval-Augmented Generation (RAG)** pipeline for enterprise document Q&A with:

- **ChromaDB** as the vector store
- **Role-Based Access Control (RBAC)** via metadata filtering
- **Guardrails AI** for PII detection & unsafe-output prevention
- **LangChain** orchestration + **Google Gemini** as the LLM

Designed for compliance-sensitive environments (healthcare / finance).

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-huggingface chromadb guardrails-ai sentence-transformers

## 2. Imports

In [ ]:
import os, re, json
from typing import Optional

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

print("All imports successful ✓")

## 3. API Key Setup

In [ ]:
import os

# If running on Google Colab
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("Loaded API key from Colab secrets ✓")
except Exception:
    os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY_HERE"
    print("Set API key manually — replace the placeholder above.")

## 4. Synthetic Enterprise Documents

Documents tagged with `access_level`: `public`, `confidential`, or `top_secret`.

In [ ]:
raw_docs = [
    # PUBLIC
    Document(
        page_content=(
            "Employee Onboarding Guide: Welcome to Acme Corp. "
            "All new employees must complete HSE training within 30 days. "
            "Contact HR at hr@acmecorp.com for onboarding queries."
        ),
        metadata={"source": "hr_onboarding.txt", "access_level": "public", "department": "HR"}
    ),
    Document(
        page_content=(
            "IT Security Policy: Employees must use strong passwords (min 12 chars) "
            "and enable MFA on all corporate accounts. "
            "Report phishing emails to security@acmecorp.com immediately."
        ),
        metadata={"source": "it_policy.txt", "access_level": "public", "department": "IT"}
    ),
    Document(
        page_content=(
            "Leave Policy: Employees are entitled to 20 days paid annual leave. "
            "Leave requests must be submitted via the HR portal at least 2 weeks in advance. "
            "Sick leave requires a medical certificate after 3 consecutive days."
        ),
        metadata={"source": "leave_policy.txt", "access_level": "public", "department": "HR"}
    ),

    # CONFIDENTIAL
    Document(
        page_content=(
            "Q3 Financial Report (Internal): Revenue for Q3 was $4.2M, "
            "up 18% YoY. Operating costs rose by 6% due to headcount expansion. "
            "EBITDA margin improved to 22%. This document is confidential."
        ),
        metadata={"source": "q3_finance.txt", "access_level": "confidential", "department": "Finance"}
    ),
    Document(
        page_content=(
            "Product Roadmap 2025 (Confidential): Q1 — launch v2.0 analytics module. "
            "Q2 — integrate AI-powered anomaly detection. "
            "Q3 — enterprise SSO support. Q4 — HIPAA compliance certification."
        ),
        metadata={"source": "roadmap_2025.txt", "access_level": "confidential", "department": "Product"}
    ),
    Document(
        page_content=(
            "Salary Bands 2024 (Confidential): Junior Engineer: $70k-$90k. "
            "Senior Engineer: $110k-$140k. Engineering Manager: $150k-$180k. "
            "VP Engineering: $200k-$250k. Not for external distribution."
        ),
        metadata={"source": "salary_bands.txt", "access_level": "confidential", "department": "HR"}
    ),

    # TOP SECRET
    Document(
        page_content=(
            "M&A Target Analysis (TOP SECRET): Acme Corp is evaluating acquisition of "
            "HealthSync Inc (valuation $120M). Due diligence underway. "
            "Deal expected to close in Q2 2025. Strictly restricted to C-Suite."
        ),
        metadata={"source": "ma_analysis.txt", "access_level": "top_secret", "department": "Strategy"}
    ),
    Document(
        page_content=(
            "Patient Data Summary (TOP SECRET / HIPAA): "
            "Patient John Doe (DOB: 01/15/1980, SSN: 123-45-6789) diagnosed with "
            "Type 2 Diabetes. Treatment: Metformin 500mg twice daily. "
            "Contact: john.doe@email.com, Phone: 555-867-5309."
        ),
        metadata={"source": "patient_record.txt", "access_level": "top_secret", "department": "Healthcare"}
    ),
]

print(f"Loaded {len(raw_docs)} enterprise documents")
for doc in raw_docs:
    print(f"  [{doc.metadata['access_level'].upper():12s}] {doc.metadata['source']}")

## 5. Chunk Documents & Build ChromaDB Vector Store

In [ ]:
# Chunk
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
print(f"Split into {len(chunks)} chunks")

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("Embeddings model loaded ✓")

# ChromaDB
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="enterprise_docs",
    persist_directory="./chroma_enterprise_store"
)
vector_db.persist()
print(f"ChromaDB collection created with {vector_db._collection.count()} vectors ✓")

## 6. RBAC — Role-Based Access Control

Each role maps to a set of allowed `access_level` values. ChromaDB's `where` filter enforces this at retrieval time — users simply cannot retrieve documents above their clearance level.

In [ ]:
ROLE_PERMISSIONS = {
    "intern":    ["public"],
    "employee":  ["public"],
    "manager":   ["public", "confidential"],
    "executive": ["public", "confidential", "top_secret"],
    "admin":     ["public", "confidential", "top_secret"],
}

def get_rbac_filter(role: str) -> dict:
    """Build a ChromaDB metadata filter for the given role."""
    allowed = ROLE_PERMISSIONS.get(role, ["public"])
    if len(allowed) == 1:
        return {"access_level": {"$eq": allowed[0]}}
    return {"access_level": {"$in": allowed}}

print("RBAC role permissions:")
for role in ROLE_PERMISSIONS:
    print(f"  {role:<12} -> {ROLE_PERMISSIONS[role]}")
    print(f"  {'':12}    filter: {get_rbac_filter(role)}")

## 7. Guardrails — PII Detection & Unsafe Content Filter

Two-stage pipeline:
1. **PII Redaction** — regex patterns for SSN, phone, email, DOB, names
2. **Unsafe Content Check** — keyword-based blocking for harmful responses

In [ ]:
PII_PATTERNS = {
    "SSN":   r"\b\d{3}-\d{2}-\d{4}\b",
    "Phone": r"\b\d{3}[-.]\d{3}[-.]\d{4}\b",
    "Email": r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+",
    "DOB":   r"\b\d{2}/\d{2}/\d{4}\b",
    "Name":  r"\b(Patient|Mr|Mrs|Ms|Dr)\.?\s+[A-Z][a-z]+\s+[A-Z][a-z]+\b",
}

UNSAFE_KEYWORDS = [
    "bomb", "weapon", "kill", "hack", "exploit", "malware",
    "suicide", "self-harm", "illegal", "fraud",
]

def redact_pii(text: str):
    """Replace PII with [REDACTED-TYPE] tags."""
    findings = []
    for label, pattern in PII_PATTERNS.items():
        matches = re.findall(pattern, text)
        if matches:
            findings.append(f"{label}: {len(matches)} instance(s)")
            text = re.sub(pattern, f"[REDACTED-{label}]", text)
    return text, findings

def check_unsafe(text: str):
    """Return flagged unsafe keywords."""
    lower = text.lower()
    return [kw for kw in UNSAFE_KEYWORDS if kw in lower]

def apply_guardrails(response_text: str) -> dict:
    """
    Full guardrails pipeline.
    Returns: {output: str, audit: dict}
    """
    audit = {
        "original_length": len(response_text),
        "pii_findings": [],
        "unsafe_flags": [],
        "blocked": False
    }

    # Step 1: PII redaction
    clean_text, pii_findings = redact_pii(response_text)
    audit["pii_findings"] = pii_findings

    # Step 2: Unsafe content
    unsafe_flags = check_unsafe(clean_text)
    audit["unsafe_flags"] = unsafe_flags

    if unsafe_flags:
        audit["blocked"] = True
        clean_text = "[RESPONSE BLOCKED: Unsafe content detected by guardrails]"

    return {"output": clean_text, "audit": audit}

print("Guardrails pipeline defined ✓")

# Quick unit test
test_input = "Patient John Doe (SSN: 123-45-6789, Email: john@example.com) has Type 2 Diabetes."
result = apply_guardrails(test_input)
print(f"\nUnit test:")
print(f"  Input : {test_input}")
print(f"  Output: {result['output']}")
print(f"  Audit : {result['audit']}")

## 8. LLM Setup (Google Gemini)

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
print("LLM loaded ✓")

system_prompt = (
    "You are a secure enterprise AI assistant. "
    "Answer ONLY using the context retrieved below. "
    "If the context is insufficient, say: I don't have access to that information. "
    "Never fabricate facts. Be concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

## 9. RBAC-Aware RAG Query Function

Pipeline: `User Query + Role` → `RBAC Filter` → `ChromaDB Retrieval` → `Gemini LLM` → `Guardrails` → `Safe Answer`

In [ ]:
def enterprise_rag_query(question: str, role: str, verbose: bool = True) -> dict:
    """
    Query the enterprise RAG system with RBAC and guardrails.

    Args:
        question : Natural language question
        role     : User role (intern/employee/manager/executive/admin)
        verbose  : Print detailed output

    Returns:
        dict with answer, sources, and guardrails audit
    """
    if verbose:
        print(f"\n{'='*62}")
        print(f"  Role     : {role.upper()}")
        print(f"  Question : {question}")
        print(f"  Clearance: {ROLE_PERMISSIONS.get(role, ['public'])}")
        print(f"{'='*62}")

    # 1. RBAC filter
    rbac_filter = get_rbac_filter(role)
    retriever = vector_db.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3, "filter": rbac_filter}
    )

    # 2. RAG chain
    combine_docs_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

    try:
        response = rag_chain.invoke({"input": question})
        raw_answer = response.get("answer", "No answer generated.")
        source_docs = response.get("context", [])
    except Exception as e:
        raw_answer = f"Error: {e}"
        source_docs = []

    # 3. Guardrails
    gr_result = apply_guardrails(raw_answer)
    final_answer = gr_result["output"]
    audit = gr_result["audit"]

    # 4. Source metadata
    sources = [
        {"source": d.metadata.get("source"), "access_level": d.metadata.get("access_level")}
        for d in source_docs
    ]

    if verbose:
        print(f"\n  ANSWER  : {final_answer}")
        if audit["pii_findings"]:
            print(f"  ⚠ PII   : {audit['pii_findings']}")
        if audit["blocked"]:
            print(f"  BLOCKED : unsafe keywords {audit['unsafe_flags']}")
        if sources:
            print(f"  SOURCES : {[s['source'] for s in sources]}")

    return {
        "role": role,
        "question": question,
        "answer": final_answer,
        "sources": sources,
        "guardrails_audit": audit,
    }

print("enterprise_rag_query() ready ✓")

## 10. Demo — RBAC in Action

In [ ]:
# Query 1: Intern asks a public question (should get a full answer)
q1 = enterprise_rag_query(
    question="What is the company password policy?",
    role="intern"
)

In [ ]:
# Query 2: Intern tries to access confidential finance data
# RBAC filter blocks confidential docs → LLM has no context → answers "I don't have access"
q2 = enterprise_rag_query(
    question="What was Acme Corp's Q3 revenue and EBITDA margin?",
    role="intern"
)

In [ ]:
# Query 3: Manager asks about confidential salary bands (allowed)
q3 = enterprise_rag_query(
    question="What is the salary range for a Senior Engineer?",
    role="manager"
)

In [ ]:
# Query 4: Executive queries top-secret M&A information (allowed)
q4 = enterprise_rag_query(
    question="Which company is Acme Corp planning to acquire and at what valuation?",
    role="executive"
)

In [ ]:
# Query 5: Executive queries patient record — guardrails should redact PII
q5 = enterprise_rag_query(
    question="What are the patient details and diagnosis in the healthcare records?",
    role="executive"
)

## 11. Guardrails Unit Tests

In [ ]:
print("\n── Guardrails Unit Tests ──\n")

test_cases = [
    ("Normal safe business text with no sensitive data.", False, 0),
    ("Call John at 555-123-4567 or email john@corp.com.", False, 2),
    ("Employee SSN: 987-65-4321. DOB: 03/22/1990.", False, 2),
    ("Instructions to hack and exploit the system firewall.", True, 0),
    ("Patient Mr. John Smith (SSN: 111-22-3333) has diabetes.", False, 2),
]

for text, expect_blocked, expect_pii_count in test_cases:
    result = apply_guardrails(text)
    blocked = result["audit"]["blocked"]
    pii = result["audit"]["pii_findings"]
    status = "PASS" if blocked == expect_blocked else "FAIL"
    print(f"  [{status}] blocked={blocked} | PII={pii}")
    print(f"         In : {text[:70]}")
    print(f"         Out: {result['output'][:70]}")
    print()

## 12. RBAC Access Matrix

In [ ]:
print("\n── RBAC Access Matrix ──\n")
header = f"{'Role':<14} {'public':^10} {'confidential':^14} {'top_secret':^12}"
print(header)
print("-" * len(header))
for role, perms in ROLE_PERMISSIONS.items():
    p = "  ✓  " if "public" in perms else "  ✗  "
    c = "     ✓     " if "confidential" in perms else "     ✗     "
    t = "    ✓   " if "top_secret" in perms else "    ✗   "
    print(f"{role:<14} {p:^10} {c:^14} {t:^12}")

## 13. Architecture Summary

```
User Query + Role
       │
       ▼
  RBAC Filter  ←── Role → access_level whitelist
       │
       ▼
ChromaDB Retriever  (metadata filter: access_level $in [allowed_levels])
       │
       ▼
LangChain RAG Chain  (create_retrieval_chain + create_stuff_documents_chain)
       │
       ▼
Google Gemini LLM  (gemini-1.5-flash, temperature=0)
       │
       ▼
Guardrails Pipeline
   ├── PII Detection  → redact SSN, phone, email, DOB, names  
   └── Unsafe Content → block response if harmful keywords found
       │
       ▼
Safe, Role-Filtered Answer + Audit Log
```

**Key design decisions:**
- RBAC is enforced at *retrieval* time, not at LLM time — the model never even sees restricted documents
- Guardrails run *after* LLM generation as a safety net before the answer reaches the user
- Audit logs capture every PII finding and block event for compliance reporting